In [ ]:
"""
Test Impact of downsampling on deconvolution accuracy.
- Single Photons by themselves
- Pairs adjacent to each other...
"""

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline
from scipy.signal import decimate
from scipy.fft import fft

import gc

from util.DataGen import nai_pulse
from util.Processing import discretize, td_nnlsr_deconvolve, td_deconvolve
from util.Plotting import plot_photons

In [ ]:
downsample_factor = 10
n = 4 # number of pairs to generate
bits = 24
extra_pad = -10
mag = 200

pair_spacing = np.arange(0, 10 * downsample_factor, 1).astype(int) # in upsampled rate...
print('Spacings', pair_spacing/downsample_factor)


In [ ]:
original_kernel_times, original_kernel = nai_pulse(1)
interpolated_kernel_times = np.linspace(np.min(original_kernel_times), np.max(original_kernel_times), original_kernel_times.size * downsample_factor)

og_dt = np.diff(original_kernel_times)[0]
interp_dt = np.diff(interpolated_kernel_times)[0]

spline = CubicSpline(x=original_kernel_times, y=original_kernel)
interpolated_kernel = spline(interpolated_kernel_times)

downsampled_kernel = decimate(x=interpolated_kernel, q=10)

print(interpolated_kernel.size)


In [ ]:
fig, axis = plt.subplots(1,1,figsize=(10,3), dpi=200)
axis.plot(original_kernel_times*1E6, original_kernel, color='blue', markersize='10', marker='.', linestyle='', label='Original Kernel')
axis.plot(interpolated_kernel_times*1E6, interpolated_kernel, color='red', markersize='5', marker='.', linestyle='', label='Upsampled Kernel')
axis.plot(original_kernel_times*1E6, downsampled_kernel, color='green', markersize='5', marker='.', linestyle='-', label='Downsampled Kernel')
axis.legend()
axis.set_xlabel('Microseconds')
axis.set_title('Plot of Single Response, Original and Interpolated')

fs = 40E6 # 40 MHz
trace = np.zeros(int(fs))
trace[:original_kernel.size] = original_kernel
zxx = np.abs(fft(trace))

fig, axis = plt.subplots(1,1,figsize=(7,3), dpi=200)
# axis.plot(zxx, label='Original Response')
axis.plot(zxx[1:zxx.size//2], label='Original Response')
axis.set_ylabel('Response Spectrum [mV]')
axis.set_xlabel('Response Frequency [Hz]')



In [ ]:
# Individual Photons...
# single_offsets = np.arange(downsample_factor+1)
volts_list = []
starts = []

start = 0
trace = np.zeros((3*n) * interpolated_kernel.size)
for _ in range(downsample_factor):
    
    # mag * kernel
    trace[start: start+interpolated_kernel.size] += mag * interpolated_kernel
    starts.append(start)
    volts_list.append(mag)
    start += interpolated_kernel.size + 1
    
if start < trace.size: # trim a bit
    trace = trace[:start]

trace = discretize(trace, bits=bits)
starts = np.array(starts)
volts_list = np.array(volts_list)

downsampled_trace = decimate(x=trace, q=downsample_factor)
interpolated_times = np.arange(trace.size) * interp_dt
original_times = np.arange(downsampled_trace.size) * og_dt

fig, axes = plt.subplots(2, 1, figsize=(10,4), dpi=200)
axes[0].plot(original_times, downsampled_trace, label='Trace')
axes[0].legend()
axes[0].set_xlim(0, np.max(original_times))
axes[0].set_ylabel('mV')

step = starts / downsample_factor
step = step - np.floor(step)

deconv = td_nnlsr_deconvolve(downsampled_trace, original_kernel)
gt0 = deconv > 0
ngt0 = deconv <= 0
axes[1].plot(original_times[gt0], deconv[gt0], marker='.', linestyle='', color='cyan', label='Deconvolution > 0')
axes[1].plot(original_times[ngt0], deconv[ngt0], marker='.', linestyle='', color='cyan', alpha=.01, label='Deconvolution = 0')
# axes[1].plot([-10], [-10], 'r.', label='Shift')

axes[1].set_xlim(0, np.max(original_times))
axes[1].set_ylim(-5, mag)
axes[1].set_ylabel('mV')
axes[1].legend()

axes[1].set_xlabel('Decimal Shift from Integer Sample Index')
axes[1].set_xticks(interpolated_times[starts])
axes[1].set_xticklabels(step.round(1).astype(str))

fig, axes = plt.subplots(1, 1, figsize=(10,1), dpi=200)
mask = deconv > 1
peaks = np.where(mask)[0]
peak_pairs = peaks.reshape(-1, 2).T
diff = np.diff(deconv[mask].reshape(-1, 2).T, axis=0)
axes.plot([0, np.max(original_times)], [0,0], color='blue')
axes.plot(original_times[peak_pairs[0,:]], diff.ravel(), marker='.', linestyle='', color='red')
axes.set_ylabel('dmV/dt')
axes.set_xticks(interpolated_times[starts])
axes.set_xticklabels(step.round(1).astype(str))
axes.set_xlim(0, np.max(original_times))

print('Done')



In [ ]:
deconv_gt0 = []

if pair_spacing.size < 15:
    fig, axes = plt.subplots(pair_spacing.size, 1, figsize=(8, 8), dpi=200)

for i, spacing in enumerate(pair_spacing):
    trace = np.zeros(2 * interpolated_kernel.size)
    trace[:interpolated_kernel.size] += mag * interpolated_kernel
    trace[spacing: spacing+interpolated_kernel.size] += mag * interpolated_kernel
    
    discretized_kernel = spline(original_kernel_times)
    downsampled_trace = decimate(x=trace, q=downsample_factor)
    
    interpolated_times = np.arange(trace.size) * interp_dt
    original_times = np.arange(downsampled_trace.size) * og_dt
    
    deconv = td_nnlsr_deconvolve(downsampled_trace, discretized_kernel)
    mask = deconv > 0
    gt0 = deconv[mask]
    deconv_gt0.append(gt0)
    
    if pair_spacing.size < 15:
        axis = axes[i]
        axis.plot(original_times, downsampled_trace, label='Trace')
        
        axis.plot(original_times[mask], gt0, marker='.', linestyle='', label='Deconv > 0')
        
        axis.legend()
    
    gc.collect()

print(len(deconv_gt0), pair_spacing.size)
fig, axis = plt.subplots(1, 1, figsize=(8, 6), dpi=200)
for spacing, deconv_gt0_set in zip(pair_spacing, deconv_gt0):
    n = len(deconv_gt0_set)
    plot_x = spacing * np.ones(n)
    axis.plot(plot_x/downsample_factor, deconv_gt0_set, color='cyan', marker='.', linestyle='')
    axis.plot(plot_x/downsample_factor, deconv_gt0_set, color='cyan', alpha=.1)

axis.set_title('Deconvolution Values > 0 for Close Events of Decimal Spacing')
axis.set_ylabel('mV')
axis.set_xlabel('"Fs Samples" between two events relative to sampling rate')

# textstr = '\n'.join((
#     r'$\mu=%.2f$' % (mu, ),
#     r'$\mathrm{median}=%.2f$' % (median, ),
#     r'$\sigma=%.2f$' % (sigma, )))

textstr = 'Each vertical depicts deconv > 0 subset\nof seperate traces'

# these are matplotlib.patch.Patch properties
props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
axis.text(0.3, 0.85, textstr, transform=axis.transAxes, fontsize=14,
        verticalalignment='top', bbox=props)

In [ ]:
"""
Test Regularization methods with NNLSR on downsampled and/or noisy data!
"""

In [ ]:
import numpy as np

from scipy.sparse import lil_matrix, csr_matrix, csc_matrix, coo_matrix, bsr_matrix
from scipy.optimize import nnls, least_squares
from scipy.linalg import circulant

import matplotlib.pyplot as plt

In [ ]:
def soft_l1_loss(z):
    d1 = 2 * ((1 + z)**0.5 - 1)
    d2 = 2 * ((1 + z)**(-0.5))
    d3 = -2 * ((1 + z)**(-1.5))
    return np.stack((d1,d2,d3))

def l2_loss(z):
    d1 = np.sum(z**2, keepdims=True)
    d2 = np.sum(2 * z,  keepdims=True)
    d3 = np.sum(2 * np.ones_like(z), keepdims=True)
    return np.stack((d1,d2,d3))

def l1_loss(z):
    d1 = z
    d2 = np.ones_like(z)
    d3 = np.zeros_like(z)
    return np.stack((d1,d2,d3))

def l0_loss(z):
    d1 = z**(.5)
    d2 = .5 * z ** (-.5)
    d3 = -.25 * z ** (-1.5)
    return np.stack((d1,d2,d3))
    
def ln1_loss(z):
    d1 = z**(-.5)
    d2 = -.5 * z ** (-1.5)
    d3 = .75 * z ** (-2.5)
    return np.stack((d1,d2,d3))

def nnlsr_residual(x, C, trace):
    out = trace - C @ x
    return out

def downsampled_const_trace(counts, spectrum, binenergies, total_time, true_fs,
                            upsampling_factor, kernel, mV_per_ADC, keV_per_area,
                            baseline=0, basenoise=0, bits=8, vpp=1, saturation=True, seed=42):

    # Freeze the random number seed for reproducibility:
    seed = seed
    np.random.seed(seed)
    
    # Unit scaling
    area_per_peak = np.sum(kernel)/max(kernel)
    mV_per_keV = mV_per_ADC/(keV_per_area*area_per_peak)
    energies = np.abs(np.random.choice(binenergies, p=spectrum/sum(spectrum), size=counts))
    peak_volts = energies * mV_per_keV + baseline
    v_limit = vpp * 1000

    # Define response template
    if upsampling_factor != 1:
        dt = 1 / true_fs
        dt_interp = dt /  upsampling_factor
        f_interp = true_fs * upsampling_factor
        
        original_kernel_times = np.arange(0, kernel.size) * dt
        spline = CubicSpline(x=original_kernel_times, y=kernel)
        upsampled_kernel_times = np.arange(0, kernel.size * upsampling_factor) * dt_interp
        upsampled_kernel = spline(upsampled_kernel_times)
    
        # one trace worth of data with padding to eliminate end effects
        upsampled_time = np.arange(int(total_time * f_interp) + 2*upsampled_kernel_times.size) * dt_interp
        upsampled_index = np.random.choice(np.arange(int(total_time * f_interp)), counts)
        upsampled_index = np.sort(upsampled_index) 
        upsampled_index += kernel.size # start pad
        print(upsampled_index)
        upsampled_event_times = upsampled_time[upsampled_index]
    
        # Build upsampled trace
        trace = np.zeros(upsampled_time.size + kernel.size*2) # w/ start and end pad
        #For every incident count, create a pulse and add it to the trace:
        for i, index in enumerate(upsampled_index):
            trace[ index : index + upsampled_kernel.size ] += upsampled_kernel * energies[i]
            
        # Downsample trace
        downsampled_trace = decimate(x=trace, q=upsampling_factor)
        sampletimes = np.arange(downsampled_trace.size) * dt
        photon_index = np.digitize(upsampled_event_times, bins=sampletimes, right=False) # not efficient, but clean
        
        trace = downsampled_trace
        
    else:
        dt = 1 / true_fs
        trace_index =  np.arange(int(total_time * true_fs) + 2 * kernel.size)
        sampletimes = trace_index * dt
        photon_index = np.random.choice(trace_index[:int(total_time * true_fs)], counts) + kernel.size
        photon_index = np.sort(photon_index) # unnecessary, but nice for plotting/printing
        photon_times = sampletimes[photon_index]
        
        trace = np.zeros(trace_index.size) # w/ start and end pad
        #For every incident count, create a pulse and add it to the trace:
        for i, index in enumerate(photon_index):
            trace[ index : index + kernel.size ] += kernel * energies[i]
            

    #scale to mV, add baseline and noise, and clip:
    trace = trace * mV_per_keV
    trace += baseline
    trace += np.random.randn(sampletimes.size)*basenoise
    if saturation:
        trace[trace > v_limit] = v_limit
        
    # Digitize:
    itrace = trace/vpp/v_limit*2**bits
    for i in range(len(itrace)):
        itrace[i] = float(int(itrace[i]))
    itrace = itrace*vpp*v_limit/2**bits

    return sampletimes, itrace, photon_index, energies, peak_volts
    

In [ ]:
#variables for creating constant countrate trace
countrate = 1E6
total_time = 1E-4
counts = int(countrate*total_time) #total counts incident on the detector   
keV_per_area = .147 #determined by trial and error to match energy range of instrument 
mV_per_ADC = 1000./4096.
specscale_keV = 5.0  #spectrum scaling i.e. keV/line in the spectrum file
baseline = 0 #110
basenoise = 0 #units mV
bits = 16  #use 12 for doing listmode but use 10 to compare traces to real trace files
error_thresh = 1E0
upsample_factor = 10
seed=1

NaI_Response = np.loadtxt('../original/NaI_Response',usecols=(1),dtype=float)
bins = np.loadtxt('../original/NaI_Response',usecols=(0),dtype=float)
#s = np.genfromtxt('/home/enp//Desktop/Emorpho Analysis Software and Calibration data/Emorpho Simulations/alt5SFT_noaa_plane_rough.out', usecols = (2), skip_footer=2)
spectrum = NaI_Response
binenergies = bins*1e3 #units keV 

out = downsampled_const_trace(counts=counts,
                                spectrum=spectrum,
                                binenergies=binenergies,
                                total_time=total_time,
                                true_fs=40E6,
                                upsampling_factor=upsample_factor,
                                kernel=original_kernel,
                                mV_per_ADC=mV_per_ADC,
                                keV_per_area=keV_per_area,
                                baseline=baseline,
                                basenoise=basenoise,
                                bits=bits,
                                vpp=1,
                                saturation=True,
                                seed=seed)
sampletimes, trace, photon_index, energies, peak_volts = out

true_volts_vector = np.zeros(trace.size)
true_volts_vector[photon_index] = peak_volts

In [ ]:
_, axes = plt.subplots(1, 1, figsize=(10, 4), dpi=200)
axes.plot(sampletimes*1E6, trace, label='Trace')
axes.plot(sampletimes[photon_index]*1E6, peak_volts, marker='.', linestyle='', markersize=4, 
                color='red', label='{} Event Trace [mV]'.format(photon_index.size))
axes.legend()
# axes.set_xlim(20, 40)
print(trace.size)


In [ ]:
ms = sampletimes * 1E6

error_mx = 0

_, trace_axes = plt.subplots(1, 1, figsize=(10,6), dpi=200)
_, deconv_axes = plt.subplots(1, 1, figsize=(10,6), dpi=200)
_, deconv_axes2 = plt.subplots(1, 1, figsize=(10,6), dpi=200)
# _, error_axes = plt.subplots(1, 1, figsize=(10,6), dpi=200)

trace_axes.plot(ms, trace, marker='', linestyle='-', markersize=2,
                label='{} Sample Trace [mV]'.format(trace.size))
trace_axes.plot(ms[photon_index], peak_volts, color='red', marker='.', markersize=5, linestyle='',
                label='{} Photon Peaks [mV]'.format(peak_volts.size), alpha=.3)

losses = [
      # soft_l1_loss,
      # l1_loss,
      'linear', #Qualitatively, linear is better than cauchy
      # 'soft_l1',
      # 'cauchy', # weakens outliers, but can be hard to optimize. Still seems the best of these
      # 'huber',
      ]

# @ 1E6 Countrate, 1E-4 time
# <function soft_l1_loss at 0x7e1c2bec78b0> Total Error 4603.957453924396
# <function l1_loss at 0x7e1c2c8de700> Total Error 4629.602780302479
# linear Total Error 4629.602780302479
# soft_l1 Total Error 4612.222985272397
# cauchy Total Error 4168.186301650927
# huber Total Error 4647.640270913502
    
c = np.concatenate((original_kernel, np.zeros(trace.size - original_kernel.size)), axis=0)
circ = circulant(c)
# C_jac = csc_matrix(lil_matrix(C)) # note that this shape should match the sparsity operations so no transpose...
sparsity = csc_matrix(lil_matrix((circ != 0)))
C = csc_matrix(lil_matrix(circ))
x0 = np.ones(trace.size)

deconvs = []

for loss in losses:
    
    print(str(loss))    
    
    solution = least_squares(fun=nnlsr_residual,
                             jac='cs',
                             method='trf', # use with bounds!
                             bounds=[0, np.inf],
                             args=(C, trace),
                             x0=x0,
                             x_scale=10,
                             diff_step=1,
                             loss = loss,
                             jac_sparsity=C, # Speeds things up significantly with sparse matrices
                             ftol=1E-15,
                             max_nfev=100,
                             verbose=0
                             )
    
    deconv = solution.x
    name = str(loss)[0:].split(' ')[0]
    
    new_threshold = 1
    bad_times = deconv < new_threshold
    good_times = deconv >= new_threshold
    
    deconv_axes.plot(ms, deconv, marker='', linestyle='-', markersize=2, label="{} Deconvolution".format(name))
    deconv_axes.plot(ms[good_times], deconv[good_times], marker='o', linestyle='',
                     color='orange', markersize=2, label="Good Deconv Points", alpha=1)
    
    # residual = solution.fun
    # resid_axes = deconv_axes.twinx()
    # marker = '.'
    # linestyle = '-'
    # color = 'green'
    # markersize = 2
    # deconv_axes.plot([-1000, -10001], [-1000, -1001], marker=marker, linestyle=linestyle, color=color,markersize=markersize)
    # deconv_axes.set_ylim(-1, 1.1 * np.max(trace))
    # resid_axes.plot(ms, residual, marker=marker, linestyle=linestyle, color=color,
    #                 markersize=markersize, label="Residual", alpha=.1)
    
    good_time_index = np.where(good_times)[0]
    good_time_dt = np.diff(good_time_index)
    # for t, dt, i, mag in zip(ms[good_time_index[:-1]], good_time_index[:-1],
    #                          good_time_dt, trace[good_time_index[:-1]]):
    #     print(t, dt, i, mag)
    
    use = good_time_index[:-1][good_time_dt != 1] # takes the right side of pairs
    # other options are to take left, greater, or maybe something more sophsiticated? (consider surrounding deconv?)
    
    lower = np.zeros(x0.size)
    upper = 1E-5 * np.ones(x0.size)
    x0 = upper/10
    upper[use] = np.inf
    
    # modified_C = circ
    # # modified_C[bad_times, :] = 0
    # modified_C[:, bad_times] = 0
    # print(np.sum(circ > 0), np.sum(modified_C > 0))
    # C = csc_matrix(lil_matrix(modified_C))
    
    solution = least_squares(fun=nnlsr_residual,
                         jac='cs',
                         method='trf', # use with bounds!
                         bounds=[lower, upper],
                         args=(C, trace),
                         x0=x0,
                         x_scale=10,
                         diff_step=1,
                         loss = loss,
                         jac_sparsity=C, # Speeds things up significantly with sparse matrices
                         ftol=1E-15,
                         max_nfev=100,
                         verbose=0
                         )
    deconv = solution.x
    deconvs.append(deconv)
    deconv_axes2.plot(ms, deconv, marker='.', linestyle='', markersize=4, label="{} Deconvolution".format(name))    
    
    # error = np.abs(true_volts_vector - solution.x)
    # mask = error > error_thresh
    # error_axes.plot(ms[mask], error[mask], marker = '.', linestyle='', label='{} Abs. Error'.format(name), alpha=.5)
    # 
    # if np.max(error) > error_mx:
    #     error_mx = np.max(error)
    # print('Total Error', np.sum(error))

print(np.sum(peak_volts), np.sum(deconv))

trace_axes.set_ylabel('mV')
trace_axes.set_xlabel('microseconds')
trace_axes.set_title('Trace, Downsampled by factor of {}'.format(downsample_factor))
trace_axes.legend()
trace_axes.set_xlim(60, 80)
# trace_axes.set_ylim(0, 40)

print(np.sum(trace) / np.sum(deconv))
print(np.sum(peak_volts) / np.sum(deconv))

# old_td_deconv = td_nnlsr_deconvolve(trace, kernel=original_kernel)
# deconv_axes.plot(ms, old_td_deconv, marker='', linestyle='-', markersize=2, label="Dense TD Deconv", alpha=.4)
deconv_axes.plot(ms[photon_index], peak_volts, color='red', marker='.', linestyle='',
                label='{} Photon Peaks [mV]'.format(peak_volts.size), alpha=.3)
deconv_axes.set_ylabel('mV')
deconv_axes.set_xlabel('microseconds')
deconv_axes.set_title('Deconvolution using NNLSR')
deconv_axes.legend()
deconv_axes.set_xlim(60, 80)
# deconv_axes.set_ylim(0, 40)

deconv_axes2.plot(ms[photon_index], peak_volts, color='red', marker='.', linestyle='',
                label='{} Photon Peaks [mV]'.format(peak_volts.size), alpha=.3)
deconv_axes2.set_ylabel('mV')
deconv_axes2.set_xlabel('microseconds')
deconv_axes2.set_title('Adjacency Corrected NNLSR')
deconv_axes2.legend()
deconv_axes2.set_xlim(60, 80)

# error = np.abs(true_volts_vector - old_td_deconv)
# error_axes.plot(ms, error, marker = '.', linestyle='', label='Dense TD Deconv Error', alpha=0)
# error_axes.set_ylim(1E-1, error_mx)
# error_axes.legend()
# error_axes.set_xlim(50, 80)

In [ ]:
#TODO note the other strange thing is that here, the peaks lag the true peaks, unlike in dense NNLSR which gets it right...
# What if nnlsr_sparse_weights_with_padNLSR is subtly wrong...?
#  -it was...  switched back to old circulant method

In [ ]:
# Looks like the number of good deconv poitns in a high pileup region is N photons + 1. Perhaps we should try to deconvolve in an upsampled space...?

In [ ]:
# Adjacent combined NNLSR on THOR

# Stats study of probability of local densities for both uniform and log normal trace countrates. P(adjacent?) P(mulitple adjacent?) 
# What is a problematic local density? Is it only purely adjacent values?
# https://en.wikipedia.org/wiki/Poisson_point_process
# https://en.wikipedia.org/wiki/Exponential_distribution
# https://en.wikipedia.org/wiki/Gamma_distribution